**Model: 3D CNN R(2+1)D**

In [1]:
import torch
from models.cnn_3d import R2Plus1DClassifier

# 1. Check Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# 2. Initialize Model
print("Loading R(2+1)D Model (Downloading weights... this may take a minute)...")
model = R2Plus1DClassifier(num_classes=50).to(device)

# 3. Create Dummy Input
# Important: R(2+1)D expects (Batch, 3, Frames, H, W)
# We test with Batch=4 to simulate training load
dummy_input = torch.randn(4, 3, 16, 112, 112).to(device)

# 4. Forward Pass
with torch.no_grad(): # Disable gradient calculation to save memory for test
    output = model(dummy_input)

print(f"Output Shape: {output.shape}")
# Expected: torch.Size([4, 50])

# 5. Test Feature Extraction Mode (For Fusion later)
model.extract_features = True
features = model(dummy_input)
print(f"Feature Vector Shape: {features.shape}")
# Expected: torch.Size([4, 512])

Running on: cuda
Loading R(2+1)D Model (Downloading weights... this may take a minute)...


Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to C:\Users\rajam/.cache\torch\hub\checkpoints\r2plus1d_18-91a641e6.pth
100%|███████████████████████████████████████████████████████████████████████████████| 120M/120M [00:49<00:00, 2.52MB/s]


Output Shape: torch.Size([4, 50])
Feature Vector Shape: torch.Size([4, 512])


**Model 2: Video_ViT - VideoMAE**

In [2]:
from models.video_vit import VideoViTClassifier
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize Model
print("Initializing VideoViT...")
vit_model = VideoViTClassifier(num_classes=50).to(device)

# 2. Create Dummy Input (Batch size 2 initially to be safe)
dummy_input = torch.randn(2, 3, 16, 112, 112).to(device)

# 3. Forward Pass
print("Running ViT Forward Pass...")
with torch.no_grad():
    output = vit_model(dummy_input)

print(f"ViT Output Shape: {output.shape}")

# 4. Check Feature Extraction
vit_model.extract_features = True
features = vit_model(dummy_input)
print(f"ViT Feature Shape: {features.shape}")
# Expected: (2, 768)

C:\Users\rajam\miniconda3\envs\video_fusion\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing VideoViT...
Loading VideoMAE weights (this happens once)...


C:\Users\rajam\miniconda3\envs\video_fusion\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajam\.cache\huggingface\hub\models--MCG-NJU--videomae-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regula

Running ViT Forward Pass...
ViT Output Shape: torch.Size([2, 50])
ViT Feature Shape: torch.Size([2, 768])


**Fusion Model**

In [3]:
from models.fusion_model import VideoFusionModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize Full Fusion System
print("Building Fusion Model (This loads both R2Plus1D and VideoMAE)...")
# We start with freeze_backbones=True to save memory during this test, 
# but later for training we might unfreeze.
model = VideoFusionModel(num_classes=50, freeze_backbones=False).to(device)

# 2. Create Dummy Input
dummy_input = torch.randn(2, 3, 16, 112, 112).to(device)

# 3. Forward Pass
print("Running Fusion Forward Pass...")
with torch.no_grad():
    output = model(dummy_input)

print(f"Final System Output Shape: {output.shape}")
# Expected: torch.Size([2, 50])

Building Fusion Model (This loads both R2Plus1D and VideoMAE)...
Initializing Fusion Model Components...
Loading VideoMAE weights (this happens once)...
Running Fusion Forward Pass...
Final System Output Shape: torch.Size([2, 50])
